# Free-first BDX-inspired Open Duck Mini v2 walk policy

Simulation only. No head IMU, physical motors, or robot deployment.

**Before running:** in Kaggle's right-hand Settings pane turn Internet and File persistence on, then choose T4 x2 (P100 is acceptable). Run one cell at a time — never Save & Run All, because this notebook has human review gates. Persistence is best effort, so still download each artifact ZIP before the session ends.

**Resuming after an interrupted session or a fresh import:** upload your last artifact ZIP as a private Kaggle Dataset, attach it with Add Input, then run Setup followed by the restore/status cell. Completed stages are reused, not retrained. The status cell always reports what is done and what to run next, so nothing depends on remembering what happened last session.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

# Keep Kaggle's mounted NVIDIA driver visible, but hide its system CUDA toolkit from JAX's pip wheels.
gpu_check = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True)
attached_gpu_count = sum(line.startswith("GPU ") for line in gpu_check.stdout.splitlines())
if gpu_check.returncode != 0 or attached_gpu_count < 1:
    raise RuntimeError("No NVIDIA GPU is attached. In Kaggle Settings choose a GPU accelerator, accept the restart, then rerun Setup.")
driver_dirs = [str(path) for path in (Path("/usr/local/nvidia/lib64"), Path("/usr/local/nvidia/lib")) if path.is_dir()]
if driver_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(driver_dirs)
else:
    os.environ.pop("LD_LIBRARY_PATH", None)
os.environ.setdefault("UV_LINK_MODE", "copy")
print("NVIDIA GPU preflight:", attached_gpu_count, "device(s) attached")

FORK_URL = "https://github.com/YOUR_GITHUB_USER/Open_Duck_Playground.git"
POLICY_BRANCH = "codex/free-first-bdx-policy"
EXPECTED_UPSTREAM = "b9be205ac64488c23504ca42e5ec790337adeec3"
assert "YOUR_GITHUB_USER" not in FORK_URL, "Create a GitHub fork and set FORK_URL first"
WORK = Path("/kaggle/working")
REPO = WORK / "Open_Duck_Playground"
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

subprocess.run(["apt-get", "update"], check=True)
subprocess.run(["apt-get", "install", "-y", "git-lfs"], check=True)
subprocess.run(["python", "-m", "pip", "install", "uv"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", POLICY_BRANCH, "--single-branch", FORK_URL, str(REPO)], check=True)
else:
    assert (REPO / ".git").is_dir(), f"Refusing to replace non-Git directory: {REPO}"
    subprocess.run(["git", "fetch", "origin", POLICY_BRANCH], cwd=REPO, check=True)
    # reset --hard, not checkout -B: a tracked file that was modified in place
    # (for example the fitted reference copied into playground/.../data) is left
    # untouched by checkout when its content is identical between the old and
    # new commits, so a stale data file can silently survive a branch update.
    # Training artifacts live in /kaggle/working/artifacts, never in the repo,
    # so forcing the repo back to the fetched commit is always safe here.
    subprocess.run(["git", "checkout", "-B", POLICY_BRANCH, "FETCH_HEAD"], cwd=REPO, check=True)
    subprocess.run(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO, check=True)
subprocess.run(["git", "lfs", "install"], cwd=REPO, check=True)
subprocess.run(["git", "lfs", "pull"], cwd=REPO, check=True)
recorded = (REPO / "UPSTREAM_COMMIT").read_text().strip()
assert recorded == EXPECTED_UPSTREAM, (recorded, EXPECTED_UPSTREAM)
head_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
venv_dir = REPO / ".venv"
venv_python = venv_dir / "bin" / "python"
if venv_dir.exists() and (
    not venv_python.is_file() or not os.access(venv_python, os.X_OK)
):
    print("Repairing persisted virtual environment...")
    subprocess.run(["uv", "venv", "--clear", "--python", sys.executable, str(venv_dir)], cwd=REPO, check=True)
subprocess.run(["uv", "sync", "--locked"], cwd=REPO, check=True)
jax_probe = subprocess.check_output(["uv", "run", "python", "-c", "import jax; print(jax.default_backend()); print(jax.local_device_count())"], cwd=REPO, text=True).splitlines()
backend = jax_probe[0].strip()
device_count = int(jax_probe[1])
assert backend == "gpu", f"Expected JAX GPU backend, got {backend!r}"
assert device_count >= 1, f"Expected at least one JAX device, got {device_count}"
training_stack_output = subprocess.check_output(["uv", "run", "python", "-c", "import json; from importlib.metadata import version; from mujoco_playground._src.collision import geoms_colliding; from mujoco_playground import wrapper; from mujoco_playground.config import locomotion_params; packages = ('brax', 'jax', 'jaxlib', 'jax-cuda12-plugin', 'mujoco', 'mujoco-mjx', 'playground'); print('TRAINING_STACK=' + json.dumps({name: version(name) for name in packages}, sort_keys=True))"], cwd=REPO, text=True)
stack_prefix = "TRAINING_STACK="
stack_lines = [line.removeprefix(stack_prefix) for line in training_stack_output.splitlines() if line.startswith(stack_prefix)]
assert len(stack_lines) == 1, f"Could not identify training stack in output: {training_stack_output!r}"
training_stack = json.loads(stack_lines[0])
expected_stack = {"brax": "0.13.0", "jax": "0.6.2", "jax-cuda12-plugin": "0.6.2", "jaxlib": "0.6.2", "mujoco": "3.3.3", "mujoco-mjx": "3.3.3", "playground": "0.0.5"}
assert training_stack == expected_stack, f"Expected {expected_stack}, got {training_stack}"
(ARTIFACTS / "workspace.json").write_text(json.dumps({"fork": FORK_URL, "branch": POLICY_BRANCH, "upstream": recorded, "commit": head_commit, "backend": backend, "device_count": device_count, "training_stack": training_stack}, indent=2) + "\n")
print("Ready:", backend, "devices:", device_count, "commit:", head_commit, ARTIFACTS)

In [ ]:
NEUTRAL_BENCHMARK = "01_neutral_nominal_20m"
NEUTRAL_MODERATE = "02_neutral_moderate_60m"
NEUTRAL_FULL = "03_neutral_full_220m"
STYLE_SEEDS = (201, 202, 203)
def style_stage(seed): return f"04_style_seed_{seed}_30m"

def run_stage(name, steps, stage, seed, restore=None, imitation_scale=1.0):
    output = ARTIFACTS / name
    command = ["uv", "run", "python", "scripts/run_training_stage.py", "--name", name, "--output-dir", str(output), "--steps", str(steps), "--randomization-stage", stage, "--seed", str(seed), "--imitation-reward-weight-scale", str(imitation_scale)]
    if restore: command += ["--restore", str(restore)]
    subprocess.run(command, cwd=REPO, check=True)
    return json.loads((output / "stage_result.json").read_text())

def load_stage(name):
    """Result of an already-completed stage, read from disk.

    Later cells must never depend on a variable left over from an earlier cell:
    a Kaggle restart clears the kernel while /kaggle/working survives, so
    "resume where I left off" has to mean reading the artifacts, not the kernel.
    """
    result = ARTIFACTS / name / "stage_result.json"
    if not result.is_file():
        raise RuntimeError(f"Stage {name} has not completed. Run status() to see what is next.")
    return json.loads(result.read_text())

def save_bundle(label):
    archive = subprocess.check_output(["python", "-c", "import shutil; print(shutil.make_archive('/kaggle/working/" + label + "', 'zip', '/kaggle/working/artifacts'))"], text=True).strip()
    print("Download before ending the session:", archive)

def status():
    """Report completed stages and the next action. Reads disk, so it is
    accurate after any restart, restore, or fresh import. Call it any time."""
    subprocess.run(["uv", "run", "python", "scripts/pipeline_status.py",
                    "--artifacts", str(ARTIFACTS)], cwd=REPO, check=True)

In [ ]:
# Restore a previous session's artifacts if a backup dataset is attached, then
# report where the pipeline actually is. Safe to run every session: it never
# overwrites artifacts that are already present, and does nothing if no backup
# is attached. Use it after any interrupted session or fresh notebook import.
subprocess.run(["uv", "run", "python", "scripts/restore_artifacts.py",
                "--artifacts", str(ARTIFACTS)], cwd=REPO, check=True)
print()
status()

## 1. Smoke test and timed benchmark

The smoke run is disposable. The 20M benchmark becomes the first 20M steps of the neutral policy if reward rises and its checkpoint/ONNX files are present.

In [ ]:
smoke = run_stage("00_smoke_1m", 1_000_000, "nominal", 100)
benchmark = run_stage(NEUTRAL_BENCHMARK, 20_000_000, "nominal", 101)
subprocess.run(["uv", "run", "python", "scripts/estimate_compute.py", "--benchmark-seconds", str(benchmark["elapsed_seconds"]), "--output", str(ARTIFACTS / "compute_decision.json")], cwd=REPO, check=True)
decision = json.loads((ARTIFACTS / "compute_decision.json").read_text())
print(decision)
save_bundle("after_benchmark")
print()
status()

Open TensorBoard logs and confirm the evaluation reward rises. Also load the exported ONNX with the evaluator. Continue on Kaggle only when recommendation is kaggle. Move the saved ZIP/checkpoint to RunPod if the estimate is 10 hours or more, Kaggle interrupts two attempts, or the quota blocks the selected run.

In [ ]:
# Reads the benchmark decision and checkpoints from disk, so this cell works
# unchanged after a restart that cleared the kernel.
decision = json.loads((ARTIFACTS / "compute_decision.json").read_text())
assert decision["recommendation"] == "kaggle", "Stop here and use the paid-resume section"
subprocess.run(["uv", "run", "python", "scripts/randomization_audit.py", "--stage", "full", "--samples", "10000", "--output", str(ARTIFACTS / "randomization_audit.json")], cwd=REPO, check=True)
moderate = run_stage(NEUTRAL_MODERATE, 60_000_000, "moderate", 102, load_stage(NEUTRAL_BENCHMARK)["checkpoint"])
full = run_stage(NEUTRAL_FULL, 220_000_000, "full", 103, moderate["checkpoint"])
save_bundle("robust_neutral_300m")
print()
status()

## 2. Original BDX-inspired reference

**Generate** (next cell) produces the eight command motions. Every one is automatically checked against the robot's real joint ranges *on this machine* (retrying with a small nudge if the solver lands on an unreachable pose), so the cell fails loudly rather than silently shipping an invalid motion — do not skip a failure here. It prints a **bundle fingerprint** identifying exactly this motion data.

**Review.** Download `reference_to_review.zip` and replay locally with `scripts/replay_bdx_reference.py` (the generator's own `replay_motion.py` has no Windows wheels and a viewer bug under WSL). Check the *style*, not the joint limits — those are already verified. Watch for the five traits: waddle, slight crouch, light bounce, deliberate foot lift, stable upper-body timing.

```
uv run python scripts/replay_bdx_reference.py -f RECORDINGS_DIR --check   # prints the fingerprint
uv run python scripts/replay_bdx_reference.py -f MOTION.json              # watch one
```

**Approve** (the cell after) fits and installs the reference. It never regenerates, and it refuses to approve a bundle whose fingerprint differs from the one you reviewed — the upstream solver is not reproducible across machines, so a regenerated bundle is not necessarily the one you watched.

In [ ]:
import shutil

GENERATOR = WORK / "Open_Duck_reference_motion_generator"
if not GENERATOR.exists(): subprocess.run(["git", "clone", "https://github.com/apirrone/Open_Duck_reference_motion_generator.git", str(GENERATOR)], check=True)
subprocess.run(["uv", "sync"], cwd=GENERATOR, check=True)
REFERENCE = ARTIFACTS / "bdx_reference"
base = ["uv", "run", "python", "scripts/prepare_bdx_reference.py", "--generator-root", str(GENERATOR), "--artifact-dir", str(REFERENCE)]

# Generation only. Approval lives in the next cell so that approving cannot
# regenerate the motions: the upstream solver can land on a different solution
# for identical input on a different machine, so a regenerated bundle is not
# necessarily the one a human reviewed.
if REFERENCE.exists(): shutil.rmtree(REFERENCE)  # never mix motions from an older style config
subprocess.run(base[:4] + ["generate"] + base[4:], cwd=REPO, check=True)  # self-checks joint ranges; fails loudly rather than shipping a bad motion
save_bundle("reference_to_review")
print("\nJoint limits are already verified. Download the ZIP and replay for STYLE, locally:")
print("  uv run python scripts/replay_bdx_reference.py -f RECORDINGS_DIR --check   # prints the fingerprint")
print("  uv run python scripts/replay_bdx_reference.py -f MOTION.json              # watch one")
print("Then set REVIEWED_FINGERPRINT in the next cell and run it.")
print()
status()

In [ ]:
# Run this only after replaying the motions locally. It approves, fits, and
# installs — it never regenerates, so what gets trained on is what you watched.
# Paste the fingerprint the local --check printed; approval fails if the bundle
# on this machine is not that one.
REVIEWED_FINGERPRINT = ""
REVIEW_NOTE = "Replayed all eight motions; crouch, foot lift and timing look stable, legs mirror correctly."

assert REVIEWED_FINGERPRINT, "Replay the motions locally first, then paste the fingerprint from --check"
subprocess.run(base[:4] + ["approve"] + base[4:] + ["--review-note", REVIEW_NOTE, "--expect-fingerprint", REVIEWED_FINGERPRINT], cwd=REPO, check=True)
subprocess.run(base[:4] + ["fit"] + base[4:], cwd=REPO, check=True)
subprocess.run(["uv", "run", "python", "scripts/install_reference_motion.py", "--source", str(REFERENCE / "polynomial_coefficients.pkl"), "--expect-entries", "8"], cwd=REPO, check=True)
save_bundle("reference_approved")
print()
status()

## 3. Style fine-tuning

Start all three seeds from the accepted robust neutral checkpoint. The first cell preflights that the fitted eight-motion reference is actually installed, then trains the three candidates. Evaluate them and do blind A/B reviews before running the second cell, which extends only the winner by another 120M steps for 150M total style training.

Leave imitation scale at 1.0; try 1.5 only if stability is retained but the reference is visibly ignored.

In [ ]:
# Preflight: the style seeds imitate the fitted reference, so prove the right
# one is installed before spending hours of GPU on it. Fails loudly if the
# repo still holds the stock 240-entry reference, or a stale/broken fit.
subprocess.run(["uv", "run", "python", "scripts/install_reference_motion.py",
                "--source", str(ARTIFACTS / "bdx_reference" / "polynomial_coefficients.pkl"),
                "--expect-entries", "8"], cwd=REPO, check=True)

neutral_checkpoint = load_stage(NEUTRAL_FULL)["checkpoint"]  # from disk: survives a restart
for seed in STYLE_SEEDS:
    run_stage(style_stage(seed), 30_000_000, "full", seed, neutral_checkpoint, 1.0)
save_bundle("style_candidates_30m")
print("\nEvaluate the three candidates and do the blind review before continuing.")
print()
status()

In [ ]:
# Run only after mass-grid evaluation and blind review have picked a winner.
WINNING_SEED = None
assert WINNING_SEED in STYLE_SEEDS, f"Choose one of {STYLE_SEEDS} after review"
winner = load_stage(style_stage(WINNING_SEED))  # from disk: survives a restart
style_final = run_stage("05_style_winner_additional_120m", 120_000_000, "full", WINNING_SEED, winner["checkpoint"], 1.0)
save_bundle("style_final_150m")
print()
status()

## 4. Acceptance

Run scripts/evaluate_mass_grid.py for the neutral and each style ONNX (20 episodes x 20 seconds). Create a blind pack with make_blind_style_review.py, fill its review form, and run check_style_acceptance.py. For export acceptance, rerun the mass-grid evaluator with 1 episode and 60 seconds; all nine cells are a superset of the three required configurations.

## Paid resume only when triggered

On RunPod, select Community Cloud only if the live RTX 4090 price is no more than US$0.50/hour. Upload the artifact ZIP, clone the same fork/commit, run scripts/paid_budget_guard.py --rate LIVE_RATE --elapsed-hours USED --planned-hours NEXT, and pass the downloaded checkpoint to --restore. Stop paid work at US$8, keep US$2 for recovery/export, download artifacts, then terminate the pod.